# EasyRAG Prompting And Answer Style

## What you will do

- compare generation prompts across answer styles
- inspect how `AnswerParam` changes the instruction block
- show how a custom answer model receives the packed prompt and style

## Why this matters

Prompting should be inspectable, not magical. If answer style changes behavior, you should be able to point at the prompt boundary where that change began.


## Source code anchors

- `easyrag.rag.generation.prompting.build_generation_prompt`
- `easyrag.rag.types.AnswerParam`
- `easyrag.rag.generation.pipeline.generate_answer`


In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.generation.packing import build_context_block
from easyrag.rag.generation.pipeline import generate_answer
from easyrag.rag.generation.prompting import build_generation_prompt


## Deterministic path

We use a fixed context block so the only thing that changes is the answer-style configuration.


In [ ]:
question = "How does EasyRAG keep answers grounded?"
selected_citations = [
    {"title": "Retrieval", "location": "docs/retrieval.md", "snippet": "Retrieval keeps query rewrite and grounded evidence visible."},
    {"title": "Policy", "location": "docs/policy.md", "snippet": "Citation-aware answers expose the evidence budget to readers."},
]
context_block = build_context_block(selected_citations)
styles = {
    "citation_aware": AnswerParam(style="citation_aware", require_citations=True),
    "extractive": AnswerParam(style="extractive", require_citations=True),
    "abstractive": AnswerParam(style="abstractive", require_citations=True),
}
prompts = {label: build_generation_prompt(question, context_block, param) for label, param in styles.items()}
for label, prompt in prompts.items():
    print(f"=== {label} ===")
    print("\n".join(prompt.splitlines()[:10]))
    print()


## Output inspection

The next cell uses a tiny custom answer model stub so you can see that the style parameter and packed prompt really reach the synthesis boundary.


In [ ]:
def style_echo_answer_model(prompt: str, **kwargs):
    return f"style={kwargs.get('style')} | citations={len(kwargs.get('citations', []))}"

prompt_tmp = tempfile.TemporaryDirectory()
prompt_rag = EasyRAG(working_dir=prompt_tmp.name, workspace="prompt-style-demo", embedding_func=_stub_embedding, query_model_func=_stub_query_model, answer_model_func=style_echo_answer_model)
run_sync(prompt_rag.initialize_storages())
run_sync(prompt_rag.ainsert("# Retrieval\nEasyRAG keeps answer synthesis grounded in citations.\n", ids=["doc::retrieval"], file_paths=["docs/retrieval.md"]))
retrieval_result = run_sync(prompt_rag.aquery(question, QueryParam(mode="naive", rewrite_enabled=False, mqe_enabled=False, chunk_top_k=3)))
for label, param in styles.items():
    answer = generate_answer(question, retrieval_result, answer_param=param, answer_model_func=prompt_rag.answer_model_func)
    _print_json({"style": label, "prompt_preview": answer.prompt.splitlines()[:8], "answer": answer.answer})


## Provider-backed path

When providers are configured, the same prompt builder still prepares the input. The optional cell prints only the prompt preview because provider answer text is less stable than the prompt contract itself.


In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_param = AnswerParam(style="citation_aware", require_citations=True)
    provider_prompt = build_generation_prompt(question, context_block, provider_param)
    print("\n".join(provider_prompt.splitlines()[:10]))


## What changed and why

The answer style changes the instruction layer, not the evidence layer. That distinction matters because a style bug should send you to the prompt builder, while a grounding bug might send you back to retrieval or citation selection.


In [ ]:
run_sync(prompt_rag.finalize_storages())
prompt_tmp.cleanup()
print("Cleaned up the prompting demo workspace.")


## Next steps

- Continue with [05_05_generation_failures_and_guardrails.ipynb](05_05_generation_failures_and_guardrails.ipynb) to see what happens when prompting or answer synthesis goes wrong.
- Read [principles/19-prompting-and-answer-synthesis.md](../../docs/principles/19-prompting-and-answer-synthesis.md) for the conceptual prompt-layer view.
- Revisit [05_03_context_assembly_and_packing.ipynb](05_03_context_assembly_and_packing.ipynb) if you want the context-block side of the same prompt contract.
